# Healthcheck Datascout Environments
## Define golden record
`GOLDEN_RECORD = "demo83"`

## Heathcheck steps
- Find all IQAs from golden record
- Find all IQAs in each client and compare against golden record

### Generalized report
❌ Error
⚠️ Differences
✅ OK
🟦 Golden Record

### Detailed report
-> Comparing 'client_id' against golden record 'client_id':
  - ➖ Missing (0):
  - ➕ Extra (0):


## Check integrity of each IQA 
- fields
- alias
    - casing
    - is empty
- sources
- filters

## Check content pages
-> Comparing 'client_id' against golden record 'client_id':
  - ➖ Missing (0):
  - ➕ Extra (0):

In [2]:
%pip install python-dotenv onepassword-sdk requests pandas importnb openai nbimporter nbformat


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
from datetime import datetime, timedelta
import pandas as pd
import time

from nbimporter import NotebookLoader
import importlib.util

import os

import requests
from requests.exceptions import HTTPError

# from document_system import get_imis_data, simplify_imis_documents,get_document_id

import ast

import sys
from pathlib import Path

# Add the root of the repo to sys.path
repo_root = Path.cwd().resolve().parents[0] 
sys.path.append(str(repo_root))

from common.onepassword_manager import OnePasswordManager
op = OnePasswordManager()

# Load the IMIS client module
from common.imis_client import IMISClient

print("✅ Loaded the modules successfully.")

✅ Loaded the modules successfully.


In [2]:
#client_ids = ["demo83", "isgdemo106", "armdemo96", "atdemo81", "i8vdemo13", "imisdemo11", "atdemo2", 
#            "apimisdemo25", "ensyncdemo13", "isgdemo14", "ccs", "imis87", "aaae", "imis36", "atsdemo89", 
#           "ibcdemo80", "imis34", "cpanb", "imis104", "oasw"]

client_ids = [
    "demo83",
    "aaae",
    "atsdemo89",
    "aboncle",
    "armdemo96",
    "imisdemo11",
    "imis36",
    "apimisdemo25",
    "imis87",
    "imis104",
    "demosales3",
    "demosales50",
    "atdemo2",
    "atdemo81",
    "demo42",
    "bsidemo27",
    "ccs",
    "demo86",
    "ensyncdemo13",
    "i8vdemo13",
    "ibcdemo80",
    "isgdemo14",
    "isgdemo106",
    "livediff",
    "oasw",
    "cpanb"
]

environment_credentials = {}
for client in client_ids:
    # --- Fetch & flatten ---
    secrets = await op.get_flattened_client_item(client)

    IMIS_BASE_URL = secrets.get("imis_base_url")
    IMIS_USERNAME = secrets.get("username")
    IMIS_PWD = secrets.get("password")

    environment_credentials[client] = {
        "client_id": client,
        "imis_base_url": IMIS_BASE_URL,
        "imis_username": IMIS_USERNAME,
        "imis_pwd": IMIS_PWD
    }

## Set the golden record here

In [3]:
# Golden Record
GOLDEN_RECORD = "demo83"
creds = environment_credentials[GOLDEN_RECORD]

IMIS_BASE_URL = creds["imis_base_url"]
IMIS_USERNAME = creds["imis_username"]
IMIS_PWD = creds["imis_pwd"]

print(f"\n🔍 Running IQA checks for: {GOLDEN_RECORD}")

imis_client = IMISClient(IMIS_BASE_URL, IMIS_USERNAME, IMIS_PWD)




🔍 Running IQA checks for: demo83
ℹ️[IMISClient] Token expires at 2025-11-07 11:23:31.344045
🟢[IMISClient] Token successfully retrieved


In [4]:
# Root folder — all IQAs are discovered dynamically from here (recursive)
IQA_ROOT = "$/_DataScout"
CONTENT_PATH = "@/Shared_Content/Datascout"

print(f"\n📂 Discovering all IQAs under: {IQA_ROOT}")
all_docs = imis_client.list_iqas_in_folder(IQA_ROOT)
iqa_list = [
    doc for doc in all_docs
    if doc.get("Type") == "IQD"
    and not (doc.get("Path") or "").startswith("$/_DataScout/Demo")
]

print(f"\n✅ Found {len(iqa_list)} IQAs:")
for doc in iqa_list:
    print(f"  {doc['Path']}")



📂 Listing documents in folder: $/_DataScout/AiParts/Profile
Document Path: $/_DataScout/AiParts/Profile/Datascout AiParts - Profile Security, Document Name: Datascout AiParts - Profile Security
Document Path: $/_DataScout/AiParts/Profile/association_average_revenue, Document Name: association_average_revenue
Document Path: $/_DataScout/AiParts/Profile/properties_by_id, Document Name: properties_by_id
Document Path: $/_DataScout/AiParts/Profile/profile_completeness_by_id, Document Name: profile_completeness_by_id
Document Path: $/_DataScout/AiParts/Profile/member_type_lookup, Document Name: member_type_lookup
Document Path: $/_DataScout/AiParts/Profile/event_registrations_by_id, Document Name: event_registrations_by_id
Document Path: $/_DataScout/AiParts/Profile/shared_tags, Document Name: shared_tags
Document Path: $/_DataScout/AiParts/Profile/changelog_by_id, Document Name: changelog_by_id
Document Path: $/_DataScout/AiParts/Profile/contact_by_id, Document Name: contact_by_id
Documen

In [5]:
iqa_path_list = iqa_list 

#  Extract the "Path" values from that list
iqa_paths = [item["Path"] for item in iqa_path_list if "Path" in item]
iqa_paths

['$/_DataScout/AiParts/Profile/Datascout AiParts - Profile Security',
 '$/_DataScout/AiParts/Profile/association_average_revenue',
 '$/_DataScout/AiParts/Profile/properties_by_id',
 '$/_DataScout/AiParts/Profile/profile_completeness_by_id',
 '$/_DataScout/AiParts/Profile/member_type_lookup',
 '$/_DataScout/AiParts/Profile/event_registrations_by_id',
 '$/_DataScout/AiParts/Profile/shared_tags',
 '$/_DataScout/AiParts/Profile/changelog_by_id',
 '$/_DataScout/AiParts/Profile/contact_by_id',
 '$/_DataScout/AiParts/Profile/category_lookup',
 '$/_DataScout/AiParts/Profile/tags_by_id',
 '$/_DataScout/AiParts/Profile/contact_extension_by_id',
 '$/_DataScout/AiParts/Profile/activitylog_by_id',
 '$/_DataScout/AiParts/Profile/committees_by_id',
 '$/_DataScout/AiParts/Profile/invoices_by_id',
 '$/_DataScout/Segments/tags_added_by_date',
 '$/_DataScout/Segments/tags_deleted_by_date',
 '$/_DataScout/Segments/tag_refresh_name_log',
 '$/_DataScout/Segments/tag_refresh_tenure',
 '$/_DataScout/Segments/

In [6]:
# Setup
full_results = []
errors = {}
clients = {}
base_df = clients.get(GOLDEN_RECORD)

# Health check logic
def check_iqa_endpoint(imis_client, iqa_path, limit=5, retries=3, retry_delay=2):
    attempt = 0
    while attempt < retries:
        try:
            start_time = time.time()
            data = imis_client.fetch_iqa(iqa_path, limit=limit)
            duration = time.time() - start_time

            print(f"✅ Fetched {len(data)} records from {iqa_path} in {round(duration, 2)} seconds.")
            print(data)

            status = "✅ Success"
            status_detail = ""

            return {
                "status": status,
                "status_detail": status_detail,
                "total_count": len(data),
                "duration_sec": round(duration, 2)
            }

        except Exception as e:
            print(f"🔥🔥🔥🔥🔥 Error on attempt {attempt + 1} for path {iqa_path}: {e}")
            attempt += 1
            if attempt >= retries:
                return {
                    "status": "❌",
                    "status_detail": str(e),
                    "total_count": 0,
                    "duration_sec": 0
                }
            time.sleep(retry_delay)

# Helpers
def status_emoji(status):
    return "✅ Success" if "Success" in status else "❌ Error"


Check each IQA in each environemnt

In [7]:
# ⏱️ Start the global timer
global_start_time = time.time()

# Run per client
for account_name, creds in environment_credentials.items():
    IMIS_BASE_URL = creds["imis_base_url"]
    IMIS_USERNAME = creds["imis_username"]
    IMIS_PWD = creds["imis_pwd"]

    print(f"\n🔍 Running IQA checks for: {account_name}")

    try:
        imis_client = IMISClient(IMIS_BASE_URL, IMIS_USERNAME, IMIS_PWD)
    except Exception as e:
        error_msg = f"Login failed: {e}"
        print(f"❌ {account_name}: {error_msg}")
        errors[account_name] = error_msg
        continue

    client_results = []

    for path in iqa_paths:
        print(f"🔍 - Checking IQA Path: {path}")
        result = check_iqa_endpoint(imis_client, path)
        client_results.append({
            "Account": account_name,
            "Path": path,
            "Status": status_emoji(result["status"]),
            "StatusDetail": result["status_detail"],
            "TotalCount": result["total_count"],
            "DurationSec": result["duration_sec"]
        })

    # Create per-account DataFrame
    df_account = pd.DataFrame(client_results)

    clients[account_name] = df_account

    # ✅ Convert TotalCount to Int64 only if it exists
    if "TotalCount" in df_account.columns:
        df_account["TotalCount"] = df_account["TotalCount"].astype("Int64")

    #df_account["TotalCount"] = df_account["TotalCount"].astype("Int64")
    # print(f"\n📄 Results for {account_name}:")
    # print(df_account.to_string(index=False))

    # Save to full list
    full_results.extend(client_results)


# ⏱️ End the timer after all accounts are processed
global_duration = round(time.time() - global_start_time, 2)


🔍 Running IQA checks for: demo83
ℹ️[IMISClient] Token expires at 2025-11-07 11:23:56.579979
🟢[IMISClient] Token successfully retrieved
🔍 - Checking IQA Path: $/_DataScout/AiParts/Profile/Datascout AiParts - Profile Security
ℹ️ Info: 🔴[IMISClient] Zero items returned at offset 0, breaking to avoid infinite loop.
ℹ️ Info: 🟢[IMISClient] Completed fetching in 0.88s. Total records fetched: 0
✅ Fetched 0 records from $/_DataScout/AiParts/Profile/Datascout AiParts - Profile Security in 0.88 seconds.
[]
🔍 - Checking IQA Path: $/_DataScout/AiParts/Profile/association_average_revenue
ℹ️ Info: Fetched 1/1 (100.00%) in 0.88s this batch.
ℹ️ Info: 🟢[IMISClient] Completed fetching in 0.88s. Total records fetched: 1
✅ Fetched 1 records from $/_DataScout/AiParts/Profile/association_average_revenue in 0.88 seconds.
[{'average_lifetime_value': 1146.58002, 'member_average_yearly_revenue': 91.875592}]
🔍 - Checking IQA Path: $/_DataScout/AiParts/Profile/properties_by_id
ℹ️ Info: Fetched 5/60 (8.33%) in 0.7

In [8]:
# Combine all account results
df_all = pd.DataFrame(full_results)

# Safely convert TotalCount
if "TotalCount" in df_all.columns:
    df_all["TotalCount"] = df_all["TotalCount"].astype("Int64")

# Separate problem rows and success rows
df_problems = df_all[df_all["Status"] != "✅"].copy()
df_successes = df_all[df_all["Status"] == "✅"].copy()

# Combine: problems first, then successes
df_final = pd.concat([df_problems, df_successes], ignore_index=True)

# Display the final, ordered result
# Print or log total duration
print(f"\n⏱️ Full health check process duration: {global_duration} seconds for {len(environment_credentials)} clients")
print()
print("\n📊 Final IQA Endpoint Check:")
df_final



⏱️ Full health check process duration: 60.2 seconds for 2 clients


📊 Final IQA Endpoint Check:


,Account,Path,Status,StatusDetail,TotalCount,DurationSec
0,demo83,$/_DataScout/AiParts/Profile/Datascout AiParts...,✅ Success,,0,0.88
1,demo83,$/_DataScout/AiParts/Profile/association_avera...,✅ Success,,1,0.88
2,demo83,$/_DataScout/AiParts/Profile/properties_by_id,✅ Success,,5,0.70
3,demo83,$/_DataScout/AiParts/Profile/profile_completen...,✅ Success,,5,1.25
4,demo83,$/_DataScout/AiParts/Profile/member_type_lookup,✅ Success,,5,0.61
5,demo83,$/_DataScout/AiParts/Profile/event_registratio...,✅ Success,,5,1.03
6,demo83,$/_DataScout/AiParts/Profile/shared_tags,✅ Success,,5,0.71
7,demo83,$/_DataScout/AiParts/Profile/changelog_by_id,✅ Success,,5,0.73
8,demo83,$/_DataScout/AiParts/Profile/contact_by_id,✅ Success,,5,3.78
9,demo83,$/_DataScout/AiParts/Profile/category_lookup,✅ Success,,3,0.62


In [9]:
df_all

,Account,Path,Status,StatusDetail,TotalCount,DurationSec
0,demo83,$/_DataScout/AiParts/Profile/Datascout AiParts...,✅ Success,,0,0.88
1,demo83,$/_DataScout/AiParts/Profile/association_avera...,✅ Success,,1,0.88
2,demo83,$/_DataScout/AiParts/Profile/properties_by_id,✅ Success,,5,0.70
3,demo83,$/_DataScout/AiParts/Profile/profile_completen...,✅ Success,,5,1.25
4,demo83,$/_DataScout/AiParts/Profile/member_type_lookup,✅ Success,,5,0.61
5,demo83,$/_DataScout/AiParts/Profile/event_registratio...,✅ Success,,5,1.03
6,demo83,$/_DataScout/AiParts/Profile/shared_tags,✅ Success,,5,0.71
7,demo83,$/_DataScout/AiParts/Profile/changelog_by_id,✅ Success,,5,0.73
8,demo83,$/_DataScout/AiParts/Profile/contact_by_id,✅ Success,,5,3.78
9,demo83,$/_DataScout/AiParts/Profile/category_lookup,✅ Success,,3,0.62


In [10]:
# === Final filtered view with only issues ===
#df_all["TotalCount"] = df_all["TotalCount"].astype("Int64")
df_errors_only = df_all[df_all["Status"] == "❌ Error"].copy()

print("\n🚨 Problematic IQA Endpoints Across All Clients:")
df_errors_only


🚨 Problematic IQA Endpoints Across All Clients:


,Account,Path,Status,StatusDetail,TotalCount,DurationSec


# Run code for slack messages

## IQA endpoints + metadata

In [11]:
import pandas as pd
import json

def extract_metadata_as_dict(iqa_json, path):
    """Extract and flatten metadata, keeping nested summaries for properties, sources, and joins."""
    result = iqa_json.get("Result", {})
    document = result.get("Document", {})

    # Extract base fields
    metadata = {
        "Path": path,
        "Name": document.get("Name"),
        "Description": document.get("Description"),
    }

    # --- Flatten Properties ---
    props = result.get("Properties", {}).get("$values", [])
    simplified_props = []
    for prop in props:
        simplified_props.append({
            "field": prop.get("PropertyName", ""),
            "alias": prop.get("Alias", ""),
            "source": prop.get("Caption", "").split(".")[0]  # crude but effective source extraction
        })

    # --- Flatten Sources ---
    sources = result.get("Sources", {}).get("$values", [])
    simplified_sources = [src.get("BusinessControllerName", "") for src in sources]

    # Add to metadata dictionary
    metadata["Properties"] = simplified_props
    metadata["Sources"] = simplified_sources

    return metadata

# List of metadata dicts to build final DataFrame
metadata_list = []

# Loop over each IQA path from your GOLDEN RECORD
iqa_paths = set(clients[GOLDEN_RECORD]["Path"].dropna())

for path in iqa_paths:
    result = imis_client.get_iqa_data(path)

    if result.get("IsSuccessStatusCode", False):
        metadata = extract_metadata_as_dict(result, path)
        metadata_list.append(metadata)
    else:
        print(f"⚠️ Failed to fetch metadata for: {path}")

# Create final DataFrame
df_iqa_metadata = pd.DataFrame(metadata_list)
df_iqa_metadata

AttributeError: 'IMISClient' object has no attribute 'get_iqa_data'

### Get each IQA path + metadata for each client

In [12]:
clients = {}

for account_name, creds in environment_credentials.items():
    IMIS_BASE_URL = creds["imis_base_url"]
    IMIS_USERNAME = creds["imis_username"]
    IMIS_PWD = creds["imis_pwd"]

    print(f"\nProcessing {account_name}...")

    try:
        imis_client = IMISClient(IMIS_BASE_URL, IMIS_USERNAME, IMIS_PWD)
        
        final_result = imis_client.list_iqas_in_folder(IQA_ROOT)
        df_results = pd.DataFrame(final_result)

        metadata_list = []  
        for path in df_results["Path"].dropna():
            result = imis_client.get_iqa_data(path)

            if result.get("IsSuccessStatusCode", False):
                metadata = extract_metadata_as_dict(result, path)
                metadata_list.append(metadata)
            else:
                print(f"⚠️ Failed to fetch metadata for: {path}")

        # Create final DataFrame
        df_iqa_metadata = pd.DataFrame(metadata_list)

        clients[account_name] = df_iqa_metadata

    except (HTTPError, Exception) as e:
        error_msg = f"{e}"
        if error_msg != "AttributeError: 'NoneType' object has no attribute 'json'":
            print(f"{account_name}: {error_msg}")
            errors[account_name] = error_msg




Processing demo83...
ℹ️[IMISClient] Token expires at 2025-10-30 18:32:16.958446
🟢[IMISClient] Token successfully retrieved

Processing isgdemo106...
ℹ️[IMISClient] Token expires at 2025-10-30 18:32:27.917276
🟢[IMISClient] Token successfully retrieved

Processing armdemo96...
ℹ️[IMISClient] Token expires at 2025-10-30 18:32:39.278467
🟢[IMISClient] Token successfully retrieved

Processing atdemo81...
ℹ️[IMISClient] Token expires at 2025-10-30 18:32:50.374659
🟢[IMISClient] Token successfully retrieved

Processing i8vdemo13...
ℹ️[IMISClient] Token expires at 2025-10-30 18:33:02.680135
🟢[IMISClient] Token successfully retrieved

Processing imisdemo11...
ℹ️[IMISClient] Token expires at 2025-10-30 18:33:29.662416
🟢[IMISClient] Token successfully retrieved

Processing atdemo2...
ℹ️[IMISClient] Token expires at 2025-10-30 18:33:40.628418
🟢[IMISClient] Token successfully retrieved

Processing apimisdemo25...
ℹ️[IMISClient] Token expires at 2025-10-30 18:33:53.822200
🟢[IMISClient] Token successf

### Create detailed report for each client

In [8]:
import difflib

def compare_client_to_golden(client_meta, golden_meta):
    diffs = {}

    # Compare Properties
    def normalize_props(props):
        # Ensure all keys exist and consistent casing
        return [
            {
                "field": p.get("field", ""),
                "alias": p.get("alias", ""),
                "source": p.get("source", "")
            }
            for p in props
        ]

    client_props = normalize_props(client_meta.get('Properties', []))
    golden_props = normalize_props(golden_meta.get('Properties', []))

    client_fields_set = { (p['field'], p['alias'], p['source']) for p in client_props }
    golden_fields_set = { (p['field'], p['alias'], p['source']) for p in golden_props }

    if client_fields_set != golden_fields_set:
        # Save as list of dicts for missing/extra
        diffs["Missing Fields"] = [
            {"field": f, "alias": a, "source": s}
            for (f, a, s) in golden_fields_set - client_fields_set
        ]
        diffs["Extra Fields"] = [
            {"field": f, "alias": a, "source": s}
            for (f, a, s) in client_fields_set - golden_fields_set
        ]

    # Compare Sources
    client_sources = set(client_meta.get("Sources", []))
    golden_sources = set(golden_meta.get("Sources", []))
    if client_sources != golden_sources:
        diffs["Missing Sources"] = list(golden_sources - client_sources)
        diffs["Extra Sources"] = list(client_sources - golden_sources)

    return {k: v for k, v in diffs.items() if v}  # filter empty categories


golden_meta = clients[GOLDEN_RECORD]
report = {}

for account, df in clients.items():
    if account == GOLDEN_RECORD:
        continue

    print(f"🔍 Comparing {account} to GOLDEN_RECORD")

    paths = set(df["Path"].dropna())

    for iqa in paths:
        # Get golden and client IQA metadata for the same path
        golden_iqa = golden_meta[golden_meta["Path"] == iqa]
        client_iqa = df[df["Path"] == iqa]

        if golden_iqa.empty:
            continue


        golden_row = golden_iqa.iloc[0].to_dict()
        client_row = client_iqa.iloc[0].to_dict()

        # Compare and collect differences
        differences = compare_client_to_golden(client_row, golden_row)


        if differences and differences != {}: 
            report.setdefault(account, []).append({
                "path": iqa,
                "differences": differences
            })


🔍 Comparing isgdemo106 to GOLDEN_RECORD
🔍 Comparing armdemo96 to GOLDEN_RECORD
🔍 Comparing atdemo81 to GOLDEN_RECORD
🔍 Comparing i8vdemo13 to GOLDEN_RECORD
🔍 Comparing imisdemo11 to GOLDEN_RECORD
🔍 Comparing atdemo2 to GOLDEN_RECORD
🔍 Comparing apimisdemo25 to GOLDEN_RECORD
🔍 Comparing ensyncdemo13 to GOLDEN_RECORD
🔍 Comparing isgdemo14 to GOLDEN_RECORD
🔍 Comparing ccs to GOLDEN_RECORD
🔍 Comparing imis87 to GOLDEN_RECORD
🔍 Comparing aaae to GOLDEN_RECORD
🔍 Comparing imis36 to GOLDEN_RECORD
🔍 Comparing atsdemo89 to GOLDEN_RECORD
🔍 Comparing ibcdemo80 to GOLDEN_RECORD
🔍 Comparing imis34 to GOLDEN_RECORD
🔍 Comparing cpanb to GOLDEN_RECORD
🔍 Comparing imis104 to GOLDEN_RECORD


### Create summary and each client report


In [9]:
summary_rows = []

# Add base account
summary_rows.append({
    "Account": GOLDEN_RECORD,
    "Status": "🟦 Golden Record",
    "MissingCount": 0,
    "ExtraCount": 0,
    "DifferencesCount": 0
})

# Add valid comparisons
for account, df in clients.items():
    if account == GOLDEN_RECORD:
        continue

    current_paths = set(df["Path"].dropna())

    missing = sorted(iqa_paths - current_paths)
    extra = sorted(current_paths - iqa_paths)

    status = "⚠️ Differences" if missing else "✅ OK"

    summary_rows.append({
        "Account": account,
        "Status": status,
        "MissingCount": len(missing),
        "ExtraCount": len(extra),
        "DifferencesCount": len(report[account]) if account in report else 0,
    })

# # Add accounts with errors
for account, error_msg in errors.items():
    summary_rows.append({
        "Account": account,
        "Status": "❌ Error",
        "MissingCount": None,
        "ExtraCount": None,
        "ErrorMessage": error_msg
    })

# Create summary DataFrame
summary_df = pd.DataFrame(summary_rows)

# Define custom sort order
status_order = {
    "❌ Error": 0,
    "⚠️ Differences": 1,
    "✅ OK": 2,
    "🟦 Golden Record": 3
}
summary_df["StatusOrder"] = summary_df["Status"].map(status_order)

# Sort and clean up
summary_df = summary_df.sort_values(by=["StatusOrder", "Account"]).drop(columns="StatusOrder").reset_index(drop=True)

# Convert columns to nullable integer type
summary_df["MissingCount"] = summary_df["MissingCount"].astype("Int64")
summary_df["ExtraCount"] = summary_df["ExtraCount"].astype("Int64")


# Display
print("\n📊 IQA Summary:")
summary_df

TypeError: unsupported operand type(s) for -: 'list' and 'set'

### Send report to Slack webhook

In [15]:
import requests
from datetime import datetime

LOGGING_ENDPOINT = "https://workflow.datascout.ai/webhook/datascout-alert"

# 🧠 Formats differences into a markdown-style message
def format_diff_summary(missing, extra, diffs):
    lines = []

    # Missing paths
    if not missing:
        lines.append("No missing paths.")
    else:
        lines.append(f"Missing ({len(missing)}):")
        for path in missing:
            lines.append(f"\t• {path}")

    # Extra paths
    if not extra:
        lines.append("No extra paths.")
    else:
        lines.append(f"Extra ({len(extra)}):")
        for path in extra:
            lines.append(f"\t• {path}")

    # Structural differences
    if not diffs:
        lines.append("No structural differences.")
    else:
        lines.append(f"Structural Differences ({len(diffs)}):")
        for diff in diffs:
            path = diff.get("path", "Unknown Path")
            lines.append(f"• Path: {path}")
            differences = diff.get("differences", {})

            if path != "$/_DataScout/AiParts/Profile/contact_extension_by_id":
                if "Missing Fields" in differences:
                    lines.append("  Missing Fields:")
                    for field in differences["Missing Fields"]:
                        lines.append(f"\t- {field}")

                if "Extra Fields" in differences:
                    lines.append("  Extra Fields:")
                    for field in differences["Extra Fields"]:
                        lines.append(f"\t- {field}")

                if "Extra Sources" in differences:
                    lines.append("  Extra Sources:")
                    for source in differences["Extra Sources"]:
                        lines.append(f"\t- {source}")

    return "\n".join(lines)


# 📬 Sends one audit report to your endpoint
def send_audit_report(account, status, message, component, environment="production", meta=None):
    payload = {
        "client_id": account,
        "environment": environment,
        "component": component,
        "status": status,
        "message": message,
        "meta": meta or {}
    }

    try:
        response = requests.post(LOGGING_ENDPOINT, json=payload, timeout=10)
        response.raise_for_status()
        print(f"📤 Sent {component} report for {account} — status: {status}")
    except Exception as e:
        print(f"❌ Failed to send {component} report for {account}: {e}")

# 📊 Sends summary across all clients
def send_audit_summary(summary_df, component):
    status_message = "\n".join(
        f"{row['Account']} - {row['Status']}" for _, row in summary_df.iterrows()
    )

    send_audit_report(
        account="ALL CLIENTS",
        status="info",
        message=f"{component.upper()} summary across {len(summary_df)} accounts",
        component=f"{component}-summary",
        meta={
            "status_details": "\n"+status_message,
        }
    )

# 🧾 Run per-client audit reports
def run_client_audit_reports(clients, report, expected_paths, GOLDEN_RECORD, component="iqa-audit"):
    for account, df in clients.items():
        if account == GOLDEN_RECORD:
            continue

        client_paths = set(df["Path"].dropna())
        diffs = report.get(account, [])
        missing = sorted(expected_paths - client_paths)
        extra = sorted(client_paths - expected_paths)

        has_structural_diffs = bool(diffs)
        has_path_diffs = (expected_paths != client_paths)

        if has_structural_diffs or has_path_diffs:
            meta = format_diff_summary(missing, extra, diffs)
            print(meta)
            message = f"Missing Count: {len(missing)}, Extra Count: {len(extra)}, Structural Differences: {len(diffs)}\n"

            send_audit_report(
                account=account,
                status="warning",
                message=message,
                component=component,
                meta=meta
            )


Send messages for IQAs

In [16]:
run_client_audit_reports(
    clients=clients,
    report=report,
    expected_paths=iqa_paths,
    GOLDEN_RECORD=GOLDEN_RECORD,
    component="iqa-audit"
)

send_audit_summary(summary_df, component="iqa")


Missing (9):
	• $/_DataScout/Segments/all_member_ids
	• $/_DataScout/Segments/contact_export
	• $/_DataScout/Segments/tag_refresh_age
	• $/_DataScout/Segments/tag_refresh_change_log
	• $/_DataScout/Segments/tag_refresh_name_log
	• $/_DataScout/Segments/tag_refresh_paid_through
	• $/_DataScout/Segments/tag_refresh_tenure
	• $/_DataScout/Segments/tags_added_by_date
	• $/_DataScout/Segments/tags_deleted_by_date
No extra paths.
Structural Differences (1):
• Path: $/_DataScout/AiParts/Profile/contact_by_id
  Missing Fields:
	- {'field': 'language_preference', 'alias': 'language_preference', 'source': 'DataScout_Data'}
📤 Sent iqa-audit report for apimisdemo25 — status: warning
📤 Sent iqa-summary report for ALL CLIENTS — status: info


## Content Pages

In [17]:
# Golden Record
GOLDEN_RECORD = "demo83"

CONTENT_PATH = "@/Shared_Content/Datascout"

base_paths = imis_client.list_documents_in_folder(CONTENT_PATH)

content_path_list = base_paths

#  Extract the "Path" values from that list
content_paths = [item["Path"] for item in content_path_list if "Path" in item]

content_paths

['@/Shared_Content/Datascout/DatascoutSSOStaging']

In [18]:
for account_name, creds in environment_credentials.items():
    IMIS_BASE_URL = creds["imis_base_url"]
    IMIS_USERNAME = creds["imis_username"]
    IMIS_PWD = creds["imis_pwd"]

    DOCUMENT_PATH = CONTENT_PATH

    print(f"\nProcessing {account_name}...")

    try:
        imis_client = IMISClient(IMIS_BASE_URL, IMIS_USERNAME, IMIS_PWD)
        
        final_result = imis_client.list_documents_in_folder(DOCUMENT_PATH)
        df_results = pd.DataFrame(final_result)
        df_results = pd.DataFrame(final_result)

        clients[account_name] = df_results

    except (HTTPError, Exception) as e:
        error_msg = f"{e}"
        if error_msg != "AttributeError: 'NoneType' object has no attribute 'json'":
            print(f"{account_name}: {error_msg}")
            errors[account_name] = error_msg



Processing demo83...
ℹ️[IMISClient] Token expires at 2025-10-24 12:05:21.735941
🟢[IMISClient] Token successfully retrieved

Processing apimisdemo25...
ℹ️[IMISClient] Token expires at 2025-10-24 12:05:24.369597
🟢[IMISClient] Token successfully retrieved


In [19]:
summary_rows = []
content_paths = set(clients[GOLDEN_RECORD]["Path"].dropna())

# Add base account
summary_rows.append({
    "Account": GOLDEN_RECORD,
    "Status": "🟦 Golden Record",
    "MissingCount": 0,
    "ExtraCount": 0,
    "ErrorMessage": ""
})

# Add valid comparisons
for account, df in clients.items():
    if account == GOLDEN_RECORD:
        continue

    current_paths = set(df["Path"].dropna())
    missing = sorted(content_paths - current_paths)
    extra = sorted(current_paths - content_paths)

    status = "⚠️ Differences" if missing else "✅ OK"

    summary_rows.append({
        "Account": account,
        "Status": status,
        "MissingCount": len(missing),
        "ExtraCount": len(extra),
        "ErrorMessage": ""
    })

# Add accounts with errors
for account, error_msg in errors.items():
    summary_rows.append({
        "Account": account,
        "Status": "❌ Error",
        "MissingCount": None,
        "ExtraCount": None,
        "ErrorMessage": error_msg
    })

# Create summary DataFrame
summary_df = pd.DataFrame(summary_rows)

# Define custom sort order
status_order = {
    "❌ Error": 0,
    "⚠️ Differences": 1,
    "✅ OK": 2,
    "🟦 Golden Record": 3
}
summary_df["StatusOrder"] = summary_df["Status"].map(status_order)

# Sort and clean up
summary_df = summary_df.sort_values(by=["StatusOrder", "Account"]).drop(columns="StatusOrder").reset_index(drop=True)

# Convert columns to nullable integer type
summary_df["MissingCount"] = summary_df["MissingCount"].astype("Int64")
summary_df["ExtraCount"] = summary_df["ExtraCount"].astype("Int64")


# Display
print("\n📊 Content Summary:")
summary_df



📊 Content Summary:


,Account,Status,MissingCount,ExtraCount,ErrorMessage
0,apimisdemo25,⚠️ Differences,5,0,
1,demo83,🟦 Golden Record,0,0,


In [20]:
run_client_audit_reports(
    clients=clients,
    report={},
    expected_paths=content_paths,
    GOLDEN_RECORD=GOLDEN_RECORD,
    component="content-audit"
)

send_audit_summary(summary_df, component="content")


Missing (5):
	• @/Shared_Content/Datascout/Account_Page
	• @/Shared_Content/Datascout/Datascout-Local-SSO
	• @/Shared_Content/Datascout/DatascoutSSO
	• @/Shared_Content/Datascout/SIT-Datascout-Account-Page-Staff
	• @/Shared_Content/Datascout/new_Account_Page_Staff
No extra paths.
No structural differences.
📤 Sent content-audit report for apimisdemo25 — status: warning
📤 Sent content-summary report for ALL CLIENTS — status: info
